# Analyze Eclipse Vertical Flow Performance (VFP) Data

The .VFP files are vertical flow performance tables (VFPPROD / VFPINJ). A practical workflow is to parse one table into a pandas DataFrame and then plot pressure vs rate for selected WCT / GOR (or other dimensions).

In [ ]:
# Import Python libraries
!pip install -q -U kaleido==0.2.1 -q -U skimpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 MB 7.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 387.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.0/118.0 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.2/106.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 60.8 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: jupyter-client
    Found existing installation: jupyter_client 7.4.9
    Uninstalling jupyter_client-7.4.9:
      Successfully uninstalled jupyter_client-7.4.9
  Attempting uninstall: ipykernel

In [ ]:
import os
from google.colab import drive
from pathlib import Path

# Mount Google Drive (run once per session)
drive.mount("/content/drive")

# Data directory (adjust if you move the notebook)
source = Path("/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/VFP/")
dest = Path("/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Output/Norne/")

# Create directory if it does not exist
os.makedirs(dest, exist_ok=True)

print("Source directory:", source)
print("Output directory:", dest)

Mounted at /content/drive
Source directory: /content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/VFP
Output directory: /content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Output/Norne


In [ ]:
from pathlib import Path

# All entries (files + subdirs)
#for p in source.iterdir():
#    print(p)

# Only files
for p in source.iterdir():
    if p.is_file():
        print(p)

/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/VFP/PD1.PIPE.Ecl
/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/VFP/pd2.VFP
/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/VFP/E1h.VFP
/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/VFP/VFPI.LOG
/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/VFP/GAS_PD2.VFP
/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/VFP/NEW_D2_GAS_0.00003.VFP
/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/VFP/F3H.Ecl
/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/VFP/PE1.PIPE.Ecl
/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/VFP/PE2.PIPE.Ecl
/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/VFP/PB2.PIPE.Ecl
/content/drive/MyDrive/Colab Notebooks/P

Below is a compact example for AlmostVertNew.VFP that you can adapt to the other VFP files.

## 1.0 Parse VFPPROD into a tidy DataFrame

This example assumes the standard VFPPROD layout: header line, followed by vectors for THP, WCT, GOR, then a multidimensional block of pressures indexed by their indices (I_TH, I_WCT, I_GOR, I_Q).

In [ ]:
from pathlib import Path
import re
import pandas as pd

vfp_path = Path(source / "E1h.VFP")  # change to any of your .VFP files

# Specify 'latin-1' encoding as a common fallback for non-UTF-8 files
with vfp_path.open(encoding='latin-1') as f:
    lines = [ln.rstrip("\n") for ln in f]

# Remove comments and blank lines
clean = [
    ln.strip() for ln in lines
    if ln.strip() and not ln.strip().startswith("--")
]

# Find VFPPROD start
start = None
for i, ln in enumerate(clean):
    if ln.upper().startswith("VFPPROD"):
        start = i + 1
        break
if start is None:
    raise ValueError("VFPPROD block not found")

idx = start

def read_vector():
    """Read one or more lines of numbers ending with '/'."""
    vals = []
    global idx
    while idx < len(clean):
        ln = clean[idx]
        idx += 1
        if "/" in ln:
            ln = ln.replace("/", " ")
            vals.extend([float(t) for t in ln.split()])
            break
        vals.extend([float(t) for t in ln.split()])
    return vals

# 1) Header: NFLOW, PREF, FLOTYPE, TABWT, TABGOR
# Example in file: 3 2629.7 'LIQ' 'WCT' 'GOR' /
hdr_tokens = clean[idx].replace("/", " ").split()
idx += 1
nflow = int(hdr_tokens[0])
pref = float(hdr_tokens[1])
flow_type = hdr_tokens[2].strip("'")
tab_wt = hdr_tokens[3].strip("'")
tab_gor = hdr_tokens[4].strip("'")

# 2) Flow vector (rates)
flow_vec = read_vector()

# 3) THP vector
thp_vec = read_vector()

# 4) WCT vector
wct_vec = read_vector()

# 5) GOR vector
gor_vec = read_vector()

n_q   = len(flow_vec)
n_thp = len(thp_vec)
n_wct = len(wct_vec)
n_gor = len(gor_vec)

# Skip any non-data lines (like '0 /') that may appear immediately after the vectors
# and before the actual pressure data blocks begin.
while idx < len(clean) and clean[idx].strip() == '0 /':
    idx += 1

# 6) Read pressure blocks: lines can start with "I_TH I_WCT I_GOR I_Q" followed by pressures,
#    or be continuation lines with only pressures.
records = []
current_indices_header = None # Stores (ith, iw, ig) from the line starting a block
current_q_start_index = None # Stores the starting iq from the line starting a block
all_p_vals_for_current_block = [] # Accumulates pressure values for the current (ith, iw, ig) combination

while idx < len(clean):
    ln = clean[idx]
    idx += 1

    if ln.strip() == "/":
        # End of the VFPPROD pressure block data
        # Process any remaining accumulated values before breaking
        if current_indices_header is not None and all_p_vals_for_current_block:
            ith_fixed, iw_fixed, ig_fixed = current_indices_header
            for k, p_wf_val in enumerate(all_p_vals_for_current_block):
                iq_actual = current_q_start_index + k
                if (iq_actual - 1) < n_q: # Ensure within bounds of flow_vec
                    records.append(
                        dict(
                            ith=ith_fixed, iw=iw_fixed, ig=ig_fixed, iq=iq_actual,
                            flow=flow_vec[iq_actual - 1],
                            thp=thp_vec[ith_fixed - 1],
                            wct=wct_vec[iw_fixed - 1],
                            gor=gor_vec[ig_fixed - 1],
                            p_wf=p_wf_val,
                        )
                    )
        break

    toks = ln.split()
    if not toks:
        continue

    is_new_index_line = False
    if len(toks) >= 4:
        try:
            # Check if the first four tokens can be converted to int
            temp_indices = [int(t) for t in toks[:4]]
            is_new_index_line = True
        except ValueError:
            pass # Not an index line, must be continuation of pressure values

    if is_new_index_line:
        # Before starting a new block, process any accumulated data from the previous block
        if current_indices_header is not None and all_p_vals_for_current_block:
            ith_fixed, iw_fixed, ig_fixed = current_indices_header
            for k, p_wf_val in enumerate(all_p_vals_for_current_block):
                iq_actual = current_q_start_index + k
                if (iq_actual - 1) < n_q:
                    records.append(
                        dict(
                            ith=ith_fixed, iw=iw_fixed, ig=ig_fixed, iq=iq_actual,
                            flow=flow_vec[iq_actual - 1],
                            thp=thp_vec[ith_fixed - 1],
                            wct=wct_vec[iw_fixed - 1],
                            gor=gor_vec[ig_fixed - 1],
                            p_wf=p_wf_val,
                        )
                    )

        # Start a new block: update current_indices_header, current_q_start_index and initialize pressure values
        current_indices_header = tuple(temp_indices[:3]) # ith, iw, ig are fixed for this block of Q-rates
        current_q_start_index = temp_indices[3] # This is the starting iq
        pressure_tokens = [t for t in toks[4:] if t != '/'] # Filter out '/'
        all_p_vals_for_current_block = [float(t) for t in pressure_tokens]
    else:
        # This line continues the pressure values for the current block
        if current_indices_header is None:
            raise ValueError("Pressure values continuation line found before any index line was parsed.")
        pressure_tokens = [t for t in toks if t != '/'] # Filter out '/'
        all_p_vals_for_current_block.extend([float(t) for t in pressure_tokens])

# After the loop, process the very last block of accumulated pressures if any
if current_indices_header is not None and all_p_vals_for_current_block:
    ith_fixed, iw_fixed, ig_fixed = current_indices_header
    for k, p_wf_val in enumerate(all_p_vals_for_current_block):
        iq_actual = current_q_start_index + k
        if (iq_actual - 1) < n_q:
            records.append(
                dict(
                    ith=ith_fixed, iw=iw_fixed, ig=ig_fixed, iq=iq_actual,
                    flow=flow_vec[iq_actual - 1],
                    thp=thp_vec[ith_fixed - 1],
                    wct=wct_vec[iw_fixed - 1],
                    gor=gor_vec[ig_fixed - 1],
                    p_wf=p_wf_val,
                )
            )

df = pd.DataFrame.from_records(records)
print(df.head())

   ith  iw  ig  iq    flow      thp  wct   gor     p_wf
0    1   1   1   1   100.0  31.0137  0.0  90.0  186.254
1    1   1   1   2   300.0  31.0137  0.0  90.0  139.360
2    1   1   1   3   600.0  31.0137  0.0  90.0  125.080
3    1   1   1   4  1000.0  31.0137  0.0  90.0  123.704
4    1   1   1   5  1600.0  31.0137  0.0  90.0  124.379


In [ ]:
from skimpy import skim

skim(df)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 6048   │ │ float64     │ 5     │                                                          │
│ │ Number of columns │ 9      │ │ int64       │ 4     │                                                          │
│ └───────────────────┴────────┘ └─────────────┴───────┘                                                          │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┓  │
│ ┃ column    ┃ NA   ┃ NA %   ┃ mean    ┃ sd        ┃ p0      ┃ p25     ┃ p50     ┃ p75     ┃ p100   ┃ hist    ┃  │
│ ┡━━━━━━━━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━┩  │
│ │ ith       │    0 │      0 │     4.5 │     2.291 │       1 │    2.75 │     4.5 │    6.25 │      8 │ █▄▄▄▄█  │  │
│ │ iw        │    0 │      0 │     3.5 │     1.708 │       1 │       2 │     3.5 │       5 │      6 │ ██████  │  │
│ │ ig        │    0 │      0 │       4 │         2 │       1 │       2 │       4 │       6 │      7 │ ▄▄▄▄▄█  │  │
│ │ iq        │    0 │      0 │     9.5 │     5.189 │       1 │       5 │     9.5 │      14 │     18 │ ██████  │  │
│ │ flow      │    0 │      0 │    4089 │      2945 │     100 │    1600 │    3750 │    6000 │  10000 │ █▅▆▃▃▃  │  │
│ │ thp       │    0 │      0 │   94.76 │     55.44 │   31.01 │   48.51 │   81.01 │   128.5 │    201 │ █▃▃▃▃▃  │  │
│ │ wct       │    0 │      0 │   0.425 │    0.3425 │       0 │     0.1 │   0.375 │    0.75 │   0.95 │ █▄ ▄▄▄  │  │
│ │ gor       │    0 │      0 │    1006 │      1658 │      90 │     110 │     200 │    1000 │   5000 │ █▂   ▂  │  │
│ │ p_wf      │    0 │      0 │   298.3 │     253.3 │   49.51 │   192.6 │   258.2 │   329.9 │   2951 │    █    │  │
│ └───────────┴──────┴────────┴─────────┴───────────┴─────────┴─────────┴─────────┴─────────┴────────┴─────────┘  │
╰────────────────────────────────────────────────────── End ──────────────────────────────────────────────────────╯

This produces a tidy DataFrame with columns: flow, thp, wct, gor, and p_wf (bottomhole pressure), plus integer indices. The precise interpretation of the 4D index ordering may vary by table; adjust the mapping from p_vals to ithp/iq if needed based on how many values you see per line in each table.

## 2.0 Visualize VFP curves with Plotly

Example: plot BHP vs LIQ rate at a fixed THP for several WCT slices.

In [ ]:
import plotly.express as px

# Choose a THP (nearest)
thp_target = 200.0
thp_sel = df.iloc[(df["thp"] - thp_target).abs().argmin()]["thp"]

plot_df = df[df["thp"] == thp_sel]

fig = px.line(
    plot_df,
    x="flow",
    y="p_wf",
    color="wct",
    line_group="wct",
    markers=True,
    labels=dict(flow="Liquid rate", p_wf="Bottomhole pressure", wct="WCT"),
    title=f"VFP curve at THP={thp_sel} (file: {vfp_path.name})",
)

fig.update_layout(
    template='ggplot2',
    width=900,
    height=600,
    legend=dict(
        x=0.05,          # 0 = left, 1 = right
        y=0.98,          # 0 = bottom, 1 = top
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.7)",  # semi‑transparent white
        bordercolor="black",
        borderwidth=1,
    )
)
#fig.update_yaxes(autorange="reversed")  # typical for pressure plots
fig.show()

Example: BHP vs rate colored by GOR at a fixed WCT:

In [ ]:
wct_target = 0.0
wct_sel = df.iloc[(df["wct"] - wct_target).abs().argmin()]["wct"]

plot_df2 = df[df["wct"] == wct_sel]

fig2 = px.line(
    plot_df2,
    x="flow",
    y="p_wf",
    color="gor",
    line_group="gor",
    markers=True,
    labels=dict(flow="Liquid rate", p_wf="Bottomhole pressure", gor="GOR"),
    title=f"VFP curve at WCT={wct_sel} (file: {vfp_path.name})",
)

fig2.update_layout(
    template='ggplot2',
    width=900,
    height=600,
    legend=dict(
        x=0.05,          # 0 = left, 1 = right
        y=0.98,          # 0 = bottom, 1 = top
        xanchor="left",
        yanchor="top",
        bgcolor="rgba(255,255,255,0.7)",  # semi‑transparent white
        bordercolor="black",
        borderwidth=1,
    )
)

#fig2.update_yaxes(autorange="reversed")
fig2.show()

You can repeat the same parsing function for other .VFP files (E1h.VFP, DevNew.VFP, GasProd.VFP, etc.) and compare their VFP behaviour by concatenating DataFrames with an extra table_name column and faceting or coloring by that in Plotly.